# COVID-19 Informative Tweet Risk Classification
## Transformer Full Fine-tuning

이 노트북은 기존 `Frozen Encoder + MLP` 방식의 문제점을 고친 최종 fine-tuning 버전입니다.

핵심 변경점:
- `AutoModelForSequenceClassification` 사용
- encoder를 freeze하지 않고 전체 모델을 fine-tuning
- train/valid/test split 분리
- validation macro F1 기준으로 best checkpoint 선택
- test set은 마지막에 한 번만 최종 평가

기본값은 사용자가 요청한 DistilBERT입니다.
팀원의 M1 모델과 encoder 계열을 맞추고 싶다면 `MODEL_NAME`을 `vinai/bertweet-covid19-base-cased`로 바꾸면 됩니다.

## 1. Colab setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# 처음 실행할 때만 설치하세요.
# BERTweet 계열을 사용할 가능성까지 고려해 emoji도 함께 설치합니다.
!pip install -q transformers datasets accelerate emoji==0.6.0 scikit-learn pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


## 2. Imports and seed

In [3]:
import glob
import inspect
import json
import os
import random

import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    AutoConfig,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

RANDOM_STATE = 42

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(RANDOM_STATE)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


## 3. Path and configuration

`MODEL_NAME` 선택 기준:
- 빠르고 가벼운 실험: `distilbert-base-uncased`
- 팀원 M1과 같은 코로나 트윗 특화 encoder 사용: `vinai/bertweet-covid19-base-cased`

팀원 코드는 BERTweet COVID encoder를 freeze해서 binary informative 분류 head만 학습했습니다.
이 노트북은 같은 `AutoModelForSequenceClassification` 구조를 사용하되, 위험도 3-class 분류에 맞추고 encoder freeze를 제거했습니다.

In [4]:
PROJECT_DIR = "/content/drive/MyDrive/AI@Sogang_4_M2"
DATA_PATH = f"{PROJECT_DIR}/data/raw/covid_informative_risk_dataset_all.tsv"
SAVE_DIR = f"{PROJECT_DIR}/models/risk_transformer_full_finetuning"
os.makedirs(SAVE_DIR, exist_ok=True)

# 요청한 DistilBERT full fine-tuning 기본값
MODEL_NAME = "distilbert-base-uncased"

# 팀원 M1 모델과 encoder를 맞추고 싶으면 위 줄 대신 아래 줄을 사용하세요.
# MODEL_NAME = "vinai/bertweet-covid19-base-cased"

TEXT_COL = "Text"
LABEL_COL = "Risk_Level"
LABEL2ID = {"Low": 0, "Medium": 1, "High": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
LABEL_NAMES = [ID2LABEL[i] for i in range(len(ID2LABEL))]

MAX_LENGTH = 128
BATCH_SIZE = 16
EPOCHS = 4
ENCODER_LR = 2e-5
CLASSIFIER_LR = 1e-4
WEIGHT_DECAY = 0.01

# Drive 경로에 없으면 Colab 세션에 직접 업로드한 TSV 파일을 자동 탐색합니다.
if not os.path.exists(DATA_PATH):
    candidate_paths = glob.glob("/content/*risk*.tsv") + glob.glob("/content/*.tsv")
    if candidate_paths:
        DATA_PATH = candidate_paths[0]
    else:
        raise FileNotFoundError(
            "TSV 파일을 찾지 못했습니다. DATA_PATH를 수정하거나 Colab에 TSV를 업로드하세요."
        )

print("DATA_PATH:", DATA_PATH)
print("MODEL_NAME:", MODEL_NAME)
print("SAVE_DIR:", SAVE_DIR)

DATA_PATH: /content/drive/MyDrive/AI@Sogang_4_M2/data/raw/covid_informative_risk_dataset_all.tsv
MODEL_NAME: distilbert-base-uncased
SAVE_DIR: /content/drive/MyDrive/AI@Sogang_4_M2/models/risk_transformer_full_finetuning


## 4. Load and clean dataset

In [5]:
df = pd.read_csv(DATA_PATH, sep="\t")
print("Raw shape:", df.shape)
print("Columns:", df.columns.tolist())

df[TEXT_COL] = df[TEXT_COL].astype("string").str.strip()
df = df[df[TEXT_COL].notna()]
df = df[df[TEXT_COL] != ""]

df[LABEL_COL] = df[LABEL_COL].astype("string").str.strip()
df = df[df[LABEL_COL].isin(LABEL2ID.keys())]

# 같은 tweet text가 train/test에 동시에 들어가는 것을 막기 위해 split 전에 중복 제거
df = df.drop_duplicates(subset=[TEXT_COL], keep="first")
df = df[[TEXT_COL, LABEL_COL]].copy()
df["labels"] = df[LABEL_COL].map(LABEL2ID).astype(int)

print("Clean shape:", df.shape)
print("Label counts:")
print(df[LABEL_COL].value_counts())
print("\nLabel ratio:")
print((df[LABEL_COL].value_counts(normalize=True) * 100).round(2))

Raw shape: (4689, 6)
Columns: ['Id', 'Text', 'Info_Label', 'Risk_Level', 'Risk_Label_Reason', 'Original_Split']
Clean shape: (4685, 3)
Label counts:
Risk_Level
High      2681
Medium    1181
Low        823
Name: count, dtype: Int64

Label ratio:
Risk_Level
High      57.23
Medium    25.21
Low       17.57
Name: proportion, dtype: Float64


## 5. Stratified train / valid / test split

In [6]:
train_valid_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=df["labels"],
)

train_df, valid_df = train_test_split(
    train_valid_df,
    test_size=0.125,
    random_state=RANDOM_STATE,
    stratify=train_valid_df["labels"],
)

train_df = train_df.reset_index(drop=True)
valid_df = valid_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

for name, split_df in [("train", train_df), ("valid", valid_df), ("test", test_df)]:
    print(f"{name}: {len(split_df)}")
    print(split_df[LABEL_COL].value_counts().reindex(LABEL2ID.keys()))
    print()

train: 3279
Risk_Level
Low        576
Medium     827
High      1876
Name: count, dtype: Int64

valid: 469
Risk_Level
Low        82
Medium    118
High      269
Name: count, dtype: Int64

test: 937
Risk_Level
Low       165
Medium    236
High      536
Name: count, dtype: Int64



## 6. Tokenizer and Hugging Face Dataset

In [7]:
# BERTweet 계열은 보통 use_fast=False가 안전합니다. DistilBERT는 fast tokenizer 사용 가능.
USE_FAST = False if "bertweet" in MODEL_NAME.lower() else True
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=USE_FAST)

def to_hf_dataset(split_df):
    return Dataset.from_pandas(split_df[[TEXT_COL, "labels"]], preserve_index=False)

train_dataset = to_hf_dataset(train_df)
valid_dataset = to_hf_dataset(valid_df)
test_dataset = to_hf_dataset(test_df)

def tokenize(batch):
    return tokenizer(
        batch[TEXT_COL],
        truncation=True,
        max_length=MAX_LENGTH,
    )

train_tokenized = train_dataset.map(tokenize, batched=True, remove_columns=[TEXT_COL])
valid_tokenized = valid_dataset.map(tokenize, batched=True, remove_columns=[TEXT_COL])
test_tokenized = test_dataset.map(tokenize, batched=True, remove_columns=[TEXT_COL])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/3279 [00:00<?, ? examples/s]

Map:   0%|          | 0/469 [00:00<?, ? examples/s]

Map:   0%|          | 0/937 [00:00<?, ? examples/s]

## 7. Model: full fine-tuning

기존 내 frozen encoder 코드와 다른 핵심은 다음입니다.

```python
# 기존 코드에서 했던 freeze를 하지 않음
# for parameter in model.base_model.parameters():
#     parameter.requires_grad = False
```

따라서 encoder와 classifier head가 모두 학습됩니다.

In [8]:
config = AutoConfig.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL2ID),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    config=config,
)

# Full fine-tuning 확인: base_model에도 trainable parameter가 있어야 합니다.
base_trainable = any(p.requires_grad for p in model.base_model.parameters())
all_trainable_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
all_param_count = sum(p.numel() for p in model.parameters())

print("Base encoder trainable:", base_trainable)
print(f"Trainable parameters: {all_trainable_count:,} / {all_param_count:,}")

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Base encoder trainable: True
Trainable parameters: 66,955,779 / 66,955,779


## 8. Metrics and class-weighted Trainer

High 클래스가 많고 Low/Medium이 상대적으로 적기 때문에 class-weighted cross entropy를 사용합니다.

In [9]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1, 2]),
    y=train_df["labels"].to_numpy(),
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)
print("Class weights [Low, Medium, High]:", class_weights)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
        "weighted_f1": f1_score(labels, preds, average="weighted"),
    }

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        weights = class_weights_tensor.to(logits.device)
        loss_fct = torch.nn.CrossEntropyLoss(weight=weights)
        loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

    def create_optimizer(self):
        """Use a smaller LR for the encoder and a larger LR for the classification head."""
        if self.optimizer is None:
            no_decay = ["bias", "LayerNorm.weight", "layer_norm.weight"]
            base_prefix = self.model.base_model_prefix

            optimizer_grouped_parameters = [
                {
                    "params": [
                        p for n, p in self.model.named_parameters()
                        if n.startswith(base_prefix) and not any(nd in n for nd in no_decay)
                    ],
                    "lr": ENCODER_LR,
                    "weight_decay": WEIGHT_DECAY,
                },
                {
                    "params": [
                        p for n, p in self.model.named_parameters()
                        if n.startswith(base_prefix) and any(nd in n for nd in no_decay)
                    ],
                    "lr": ENCODER_LR,
                    "weight_decay": 0.0,
                },
                {
                    "params": [
                        p for n, p in self.model.named_parameters()
                        if not n.startswith(base_prefix) and not any(nd in n for nd in no_decay)
                    ],
                    "lr": CLASSIFIER_LR,
                    "weight_decay": WEIGHT_DECAY,
                },
                {
                    "params": [
                        p for n, p in self.model.named_parameters()
                        if not n.startswith(base_prefix) and any(nd in n for nd in no_decay)
                    ],
                    "lr": CLASSIFIER_LR,
                    "weight_decay": 0.0,
                },
            ]

            # 빈 그룹 제거
            optimizer_grouped_parameters = [
                group for group in optimizer_grouped_parameters if len(group["params"]) > 0
            ]

            self.optimizer = torch.optim.AdamW(optimizer_grouped_parameters)
        return self.optimizer

Class weights [Low, Medium, High]: [1.89756944 1.3216445  0.5826226 ]


## 9. TrainingArguments compatibility helper

Transformers 버전에 따라 `evaluation_strategy` 또는 `eval_strategy` 이름이 다를 수 있어서 자동으로 맞춥니다.

In [11]:
def make_training_args(**kwargs):
    signature = inspect.signature(TrainingArguments.__init__)
    params = signature.parameters

    if "eval_strategy" in params and "evaluation_strategy" in kwargs:
        kwargs["eval_strategy"] = kwargs.pop("evaluation_strategy")
    elif "evaluation_strategy" in params and "eval_strategy" in kwargs:
        kwargs["evaluation_strategy"] = kwargs.pop("eval_strategy")

    return TrainingArguments(**kwargs)


def make_trainer(**kwargs):
    signature = inspect.signature(Trainer.__init__)
    params = signature.parameters

    if "processing_class" in params and "tokenizer" in kwargs:
        kwargs["processing_class"] = kwargs.pop("tokenizer")
    elif "tokenizer" in params and "processing_class" in kwargs:
        kwargs["tokenizer"] = kwargs.pop("processing_class")

    return WeightedTrainer(**kwargs)


training_args = make_training_args(
    output_dir=SAVE_DIR,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=ENCODER_LR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=WEIGHT_DECAY,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=2,
    report_to="none",
    seed=RANDOM_STATE,
)

trainer = make_trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=valid_tokenized,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

## 10. Train

In [12]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,0.665078,0.689076,0.765458,0.721199,0.768433
2,0.385692,0.523609,0.823028,0.796523,0.825881
3,0.252016,0.508146,0.855011,0.839031,0.855571
4,0.183522,0.536817,0.855011,0.839152,0.855249


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=820, training_loss=0.43697270939989785, metrics={'train_runtime': 152.7529, 'train_samples_per_second': 85.864, 'train_steps_per_second': 5.368, 'total_flos': 266092164206910.0, 'train_loss': 0.43697270939989785, 'epoch': 4.0})

## 11. Validation and final test evaluation

In [13]:
print("VALID result")
valid_result = trainer.evaluate(valid_tokenized)
print(valid_result)

print("\nTEST result")
test_result = trainer.evaluate(test_tokenized)
print(test_result)

test_output = trainer.predict(test_tokenized)
test_logits = test_output.predictions
test_pred = np.argmax(test_logits, axis=1)
test_labels = np.array(test_df["labels"].tolist())

print("\nClassification report:")
print(classification_report(
    test_labels,
    test_pred,
    labels=[0, 1, 2],
    target_names=LABEL_NAMES,
    zero_division=0,
))
print("Confusion matrix [Low, Medium, High]:")
print(confusion_matrix(test_labels, test_pred, labels=[0, 1, 2]))

VALID result


Training Loss,Validation Loss,Epoch,Accuracy,Macro F1,Weighted F1
0.183522,0.536817,4,0.855011,0.839152,0.855249


{'eval_loss': 0.5368171334266663, 'eval_accuracy': 0.8550106609808102, 'eval_macro_f1': 0.8391521121592551, 'eval_weighted_f1': 0.8552487619290181}

TEST result


Training Loss,Validation Loss,Epoch,Accuracy,Macro F1,Weighted F1
0.183522,0.508139,4,0.843116,0.825690,0.842065


{'eval_loss': 0.5081387758255005, 'eval_accuracy': 0.8431163287086446, 'eval_macro_f1': 0.8256895748209002, 'eval_weighted_f1': 0.8420650573756533}



Classification report:
              precision    recall  f1-score   support

         Low       0.82      0.88      0.85       165
      Medium       0.77      0.72      0.74       236
        High       0.88      0.89      0.88       536

    accuracy                           0.84       937
   macro avg       0.82      0.83      0.83       937
weighted avg       0.84      0.84      0.84       937

Confusion matrix [Low, Medium, High]:
[[146   5  14]
 [ 16 169  51]
 [ 16  45 475]]


## 12. Error analysis helper

In [14]:
analysis_df = test_df.copy()
analysis_df["pred_id"] = test_pred
analysis_df["Predicted_Risk_Level"] = analysis_df["pred_id"].map(ID2LABEL)
wrong_df = analysis_df[analysis_df["labels"] != analysis_df["pred_id"]].copy()

print("Wrong predictions:", len(wrong_df), "/", len(test_df))
wrong_df[[TEXT_COL, LABEL_COL, "Predicted_Risk_Level"]].head(20)

Wrong predictions: 147 / 937


,Text,Risk_Level,Predicted_Risk_Level
1,"Good Afternoon, I am pleased to announce that ...",High,Low
9,"Last time I checked there were 1,000 confirmed...",High,Medium
11,A Carnegie Mellon University back from spring ...,High,Medium
14,3 members of same Ohio family die of COVID-19 ...,Medium,High
40,"In #Brazil, Fear Surges Along with #Coronaviru...",Medium,High
45,Feeling lucky here in MO! MS can test 200 peop...,Medium,Low
51,My God: #America lost 4591 souls in one day fr...,Medium,High
60,Second person dies of coronavirus in UK says H...,Low,High
61,Princeton is I think the first university to s...,High,Low
65,#Update @USER Maryland @USER details new infor...,High,Medium


## 13. Save best model, tokenizer, and config

In [15]:
BEST_SAVE_DIR = f"{SAVE_DIR}/best_model"
trainer.save_model(BEST_SAVE_DIR)
tokenizer.save_pretrained(BEST_SAVE_DIR)

config_to_save = {
    "model_type": "transformer_full_finetuning",
    "model_name": MODEL_NAME,
    "label2id": LABEL2ID,
    "id2label": {str(k): v for k, v in ID2LABEL.items()},
    "text_col": TEXT_COL,
    "label_col": LABEL_COL,
    "max_length": MAX_LENGTH,
    "random_state": RANDOM_STATE,
    "split": {"train": 0.7, "valid": 0.1, "test": 0.2},
    "class_weights_low_medium_high": class_weights.tolist(),
    "valid_result": {k: float(v) for k, v in valid_result.items() if isinstance(v, (int, float))},
    "test_result": {k: float(v) for k, v in test_result.items() if isinstance(v, (int, float))},
}

with open(f"{BEST_SAVE_DIR}/risk_config.json", "w", encoding="utf-8") as f:
    json.dump(config_to_save, f, ensure_ascii=False, indent=2)

print("Saved best model to:", BEST_SAVE_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved best model to: /content/drive/MyDrive/AI@Sogang_4_M2/models/risk_transformer_full_finetuning/best_model


## 14. Single tweet prediction

In [16]:
def predict_tweet(text: str):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH,
    )
    inputs = {k: v.to(trainer.model.device) for k, v in inputs.items()}
    trainer.model.eval()
    with torch.no_grad():
        logits = trainer.model(**inputs).logits
        probs = torch.softmax(logits, dim=-1).cpu().numpy()[0]
    pred_id = int(np.argmax(probs))
    return {
        "text": text,
        "predicted_label": ID2LABEL[pred_id],
        "probabilities": {ID2LABEL[i]: float(probs[i]) for i in range(len(ID2LABEL))},
    }

predict_tweet("New COVID-19 deaths and cases were reported today in the city.")

{'text': 'New COVID-19 deaths and cases were reported today in the city.',
 'predicted_label': 'High',
 'probabilities': {'Low': 0.02484259568154812,
  'Medium': 0.000584281689953059,
  'High': 0.9745731353759766}}

## 15. Encoder 사용 방식
   - `AutoModelForSequenceClassification` 사용
   - encoder freeze 제거
   - encoder와 classification head를 함께 fine-tuning
   - valid split 기준으로 best checkpoint 선택